In [1]:
# Cell 1 — Mount Drive, install packages, setup solc
from google.colab import drive
drive.mount('/content/drive')
DRIVE_BASE = '/content/drive/MyDrive/PhishGuard'

import os, subprocess
for folder in ['data', 'data/checkpoints', 'features', 'models', 'evaluation']:
    os.makedirs(f'{DRIVE_BASE}/{folder}', exist_ok=True)

subprocess.run(['pip', 'install',
    'pyevmasm==0.2.3', 'slither-analyzer', 'py-solc-x',
    'pandas==2.2.2', 'numpy==1.26.4', 'requests==2.31.0', '-q'], check=False)

import solcx
for v in ['0.4.24', '0.4.25', '0.5.17', '0.6.12', '0.7.6', '0.8.0', '0.8.19']:
    try:
        solcx.install_solc(v, show_progress=False)
        print(f'  solc {v} ready')
    except Exception as e:
        print(f'  WARNING: solc {v} failed: {e}')

assert os.path.exists(f'{DRIVE_BASE}/data/raw_contract_data.csv'), \
    'raw_contract_data.csv not found'
print('Cell 1 ready.')


In [2]:
# Cell 2 — Constants and imports
import pandas as pd
import numpy as np
import json
import os
import re
import time
import requests
import tempfile
import shutil
import subprocess
from pyevmasm import disassemble_all

ETHERSCAN_API_KEY = 'YOUR_ETHERSCAN_API_KEY_HERE'
ETHERSCAN_BASE    = 'https://api.etherscan.io/v2/api'
CHAIN_ID          = 1
PROXY_SLOT        = '360894a13ba1a3210667c828492db98dca3e2076cc3735a920a3ca505d382bbc'
CHECKPOINT_PATH   = f'{DRIVE_BASE}/data/checkpoints/contract_features_checkpoint.json'
OUTPUT_PATH       = f'{DRIVE_BASE}/features/contract_features.csv'
SCHEMA_PATH       = f'{DRIVE_BASE}/models/contract_feature_schema.json'

FEATURE_COLS = [
    'is_verified', 'bytecode_size', 'abi_function_count',
    'external_public_function_count', 'approval_related_function_flag',
    'permit_related_function_flag', 'setApprovalForAll_flag',
    'opcode_freq_CALL', 'opcode_freq_DELEGATECALL', 'opcode_freq_SELFDESTRUCT',
    'opcode_freq_SSTORE', 'opcode_freq_JUMPI', 'external_call_sites_count',
    'has_create2', 'proxy_pattern_detected',
    'approval_then_external_call_pattern', 'approval_then_state_mutation_pattern',
    'control_flow_complexity_score', 'slither_warning_count_total',
    'slither_low_level_call_count', 'slither_access_control_issues_count'
]

assert ETHERSCAN_API_KEY != 'YOUR_ETHERSCAN_API_KEY_HERE', \
    'Replace ETHERSCAN_API_KEY with your real key before running'
print('Cell 2 constants loaded.')


In [3]:
# Cell 3 — Define all helper functions
# ── disassemble_safe ──────────────────────────────────────────
def disassemble_safe(bytecode_hex):
    if bytecode_hex == '0x' or not bytecode_hex:
        return []
    try:
        stripped = bytecode_hex[2:]
        if len(stripped) % 2 != 0:
            stripped = '0' + stripped
        return list(disassemble_all(bytes.fromhex(stripped)))
    except Exception:
        return []

# ── prepare_slither_files ─────────────────────────────────────
def prepare_slither_files(source_code_str, address):
    source   = source_code_str.strip()
    temp_dir = tempfile.mkdtemp()

    if source.startswith('{{'):
        try:
            parsed  = json.loads(source[1:-1])
            sources = parsed.get('sources', {})
            entry   = None
            for filename, content in sources.items():
                filepath = os.path.join(temp_dir, os.path.basename(filename))
                with open(filepath, 'w') as f:
                    f.write(content.get('content', ''))
                if entry is None:
                    entry = filepath
            return temp_dir, entry
        except Exception:
            shutil.rmtree(temp_dir, ignore_errors=True)
            return None, None

    if source.startswith('{') and '"language"' in source:
        try:
            parsed  = json.loads(source)
            sources = parsed.get('sources', {})
            entry   = None
            for filename, content in sources.items():
                filepath = os.path.join(temp_dir, os.path.basename(filename))
                with open(filepath, 'w') as f:
                    f.write(content.get('content', ''))
                if entry is None:
                    entry = filepath
            return temp_dir, entry
        except Exception:
            shutil.rmtree(temp_dir, ignore_errors=True)
            return None, None

    entry = os.path.join(temp_dir, f'{address}.sol')
    with open(entry, 'w') as f:
        f.write(source)
    return temp_dir, entry

# ── resolve_solc_path ─────────────────────────────────────────
def resolve_solc_path(compiler_version_str):
    """
    Returns the absolute path to a solc binary for the requested version.
    Uses py-solc-x. Falls back through same-minor → 0.8.19.
    Never calls solc-select or touches the network at analysis time.
    """
    import re, solcx
    from packaging.version import Version as V

    ver_match = re.search(r'v?([\d]+\.[\d]+\.[\d]+)', compiler_version_str)
    target_str = ver_match.group(1) if ver_match else '0.8.19'

    home = os.path.expanduser('~')

    def binary_path(v):
        p = f'{home}/.solcx/solc-v{v}'
        return p if os.path.exists(p) else None

    # 1. Exact match
    if binary_path(target_str):
        return binary_path(target_str)

    # 2. Try to install the exact version (uses cached download if available)
    try:
        solcx.install_solc(target_str, show_progress=False)
        if binary_path(target_str):
            return binary_path(target_str)
    except Exception:
        pass

    # 3. Nearest installed version in same major.minor
    try:
        target = V(target_str)
        installed = solcx.get_installed_solc_versions()
        same_minor = [v for v in installed
                      if v.major == target.major and v.minor == target.minor]
        if same_minor:
            best = str(max(same_minor))
            if binary_path(best):
                return binary_path(best)
    except Exception:
        pass

    # 4. Hard fallback: 0.8.19
    return binary_path('0.8.19')

# ── run_slither ───────────────────────────────────────────────
def run_slither(entry_file, solc_path):
    defaults = {
        'slither_warning_count_total': 0,
        'slither_low_level_call_count': 0,
        'slither_access_control_issues_count': 0
    }
    if not solc_path:
        return defaults

    runner = tempfile.mktemp(suffix='.py')
    output = tempfile.mktemp(suffix='.json')
    try:
        script = f"""
import json, os, inspect
os.environ['PATH'] = '/root/.solcx:/root/.local/bin:/usr/local/bin:' + os.environ.get('PATH', '')
try:
    from slither.slither import Slither
    from slither.detectors.abstract_detector import AbstractDetector
    import slither.detectors.all_detectors as ad

    detector_classes = [
        cls for _, cls in inspect.getmembers(ad, inspect.isclass)
        if issubclass(cls, AbstractDetector) and cls is not AbstractDetector
    ]

    sl = Slither({repr(entry_file)}, solc={repr(solc_path)})
    for d in detector_classes:
        sl.register_detector(d)
    raw = sl.run_detectors()

    # Slither 0.11.x: raw is list-of-lists, one inner list per detector
    all_findings = [f for det in raw for f in det]
    total   = sum(len(f.get('elements', [])) for f in all_findings)
    low_lvl = sum(len(f.get('elements', []))
                  for f in all_findings if 'low-level-calls' in f.get('check', ''))
    access  = sum(len(f.get('elements', []))
                  for f in all_findings
                  if 'access-control' in f.get('check', '')
                  or 'unprotected'    in f.get('check', ''))

    with open({repr(output)}, 'w') as f:
        json.dump({{'total': total, 'low_lvl': low_lvl, 'access': access}}, f)
except Exception as e:
    with open({repr(output)}, 'w') as f:
        json.dump({{'total': 0, 'low_lvl': 0, 'access': 0, 'err': str(e)}}, f)
"""
        with open(runner, 'w') as f:
            f.write(script)

        subprocess.run(['python3', runner], timeout=120, capture_output=True)

        if os.path.exists(output):
            with open(output) as f:
                data = json.load(f)
            return {
                'slither_warning_count_total':         data.get('total',   0),
                'slither_low_level_call_count':        data.get('low_lvl', 0),
                'slither_access_control_issues_count': data.get('access',  0)
            }
        return defaults
    except subprocess.TimeoutExpired:
        return defaults
    except Exception:
        return defaults
    finally:
        for f in [runner, output]:
            try:
                os.unlink(f)
            except Exception:
                pass

# ── get_slither_features ──────────────────────────────────────
def get_slither_features(address, api_key, etherscan_base, chain_id):
    defaults = {
        'slither_warning_count_total': 0,
        'slither_low_level_call_count': 0,
        'slither_access_control_issues_count': 0
    }
    try:
        r = requests.get(etherscan_base, params={
            'chainid': chain_id, 'module': 'contract',
            'action': 'getsourcecode', 'address': address,
            'apikey': api_key
        }, timeout=15)
        time.sleep(0.25)
        result = r.json().get('result', [])
        if not result or not result[0].get('SourceCode', ''):
            return defaults

        source_str   = result[0]['SourceCode']
        compiler_ver = result[0].get('CompilerVersion', 'v0.8.19')

        solc_path = resolve_solc_path(compiler_ver)
        if not solc_path:
            return defaults

        temp_dir, entry_file = prepare_slither_files(source_str, address)
        if entry_file is None:
            return defaults

        slither_result = run_slither(entry_file, solc_path)
        shutil.rmtree(temp_dir, ignore_errors=True)
        return slither_result
    except Exception:
        return defaults

print('Cell 3 helper functions defined.')


In [4]:
# Cell 4 — Define extract_contract_features(row)
def extract_contract_features(row):
    zeros = {col: 0 for col in FEATURE_COLS}
    try:
        addr         = row['address'].lower()
        bytecode_hex = str(row['bytecode_hex'])
        is_verified  = int(row['is_verified'])
        bytecode_size = (len(bytecode_hex) - 2) // 2 if bytecode_hex != '0x' else 0

        try:
            abi_list = json.loads(row['abi_json'])
        except Exception:
            abi_list = []
        functions  = [item for item in abi_list if item.get('type') == 'function']
        func_names = [f.get('name', '').lower() for f in functions]

        abi_function_count             = len(functions)
        external_public_function_count = sum(
            1 for f in functions
            if f.get('stateMutability') not in ['view', 'pure'])
        approval_related_function_flag = int(any(
            any(kw in n for kw in
                ['approve','setallowance','increaseallowance','decreaseallowance'])
            for n in func_names))
        permit_related_function_flag   = int(any('permit' in n for n in func_names))
        setApprovalForAll_flag         = int(any(
            n == 'setapprovalforall' for n in func_names))

        instructions = disassemble_safe(bytecode_hex)
        total_instr  = len(instructions)

        def opcode_freq(name):
            return sum(1 for i in instructions if i.name == name) / total_instr \
                   if total_instr > 0 else 0

        opcode_freq_CALL         = opcode_freq('CALL')
        opcode_freq_DELEGATECALL = opcode_freq('DELEGATECALL')
        opcode_freq_SELFDESTRUCT = opcode_freq('SELFDESTRUCT')
        opcode_freq_SSTORE       = opcode_freq('SSTORE')
        opcode_freq_JUMPI        = opcode_freq('JUMPI')
        external_call_sites_count = len(set(
            i.pc for i in instructions if i.name == 'CALL'))

        has_create2            = int(any(i.name == 'CREATE2' for i in instructions))
        proxy_pattern_detected = int(PROXY_SLOT in bytecode_hex.lower())

        names = [i.name for i in instructions]
        approval_then_external_call_pattern  = 0
        approval_then_state_mutation_pattern = 0
        for k in range(max(0, len(names) - 9)):
            window = set(names[k:k+10])
            if 'SLOAD' in window and 'CALL'   in window:
                approval_then_external_call_pattern  = 1
            if 'SLOAD' in window and 'SSTORE' in window:
                approval_then_state_mutation_pattern = 1
            if approval_then_external_call_pattern and \
               approval_then_state_mutation_pattern:
                break

        control_flow_complexity_score = (
            sum(1 for i in instructions if i.name == 'JUMPDEST') / bytecode_size
            if bytecode_size > 0 else 0)

        if is_verified == 1:
            slither_feats = get_slither_features(
                addr, ETHERSCAN_API_KEY, ETHERSCAN_BASE, CHAIN_ID)
        else:
            slither_feats = {
                'slither_warning_count_total': 0,
                'slither_low_level_call_count': 0,
                'slither_access_control_issues_count': 0
            }

        return {
            'is_verified':                           is_verified,
            'bytecode_size':                         bytecode_size,
            'abi_function_count':                    abi_function_count,
            'external_public_function_count':        external_public_function_count,
            'approval_related_function_flag':        approval_related_function_flag,
            'permit_related_function_flag':          permit_related_function_flag,
            'setApprovalForAll_flag':                setApprovalForAll_flag,
            'opcode_freq_CALL':                      opcode_freq_CALL,
            'opcode_freq_DELEGATECALL':              opcode_freq_DELEGATECALL,
            'opcode_freq_SELFDESTRUCT':              opcode_freq_SELFDESTRUCT,
            'opcode_freq_SSTORE':                    opcode_freq_SSTORE,
            'opcode_freq_JUMPI':                     opcode_freq_JUMPI,
            'external_call_sites_count':             external_call_sites_count,
            'has_create2':                           has_create2,
            'proxy_pattern_detected':                proxy_pattern_detected,
            'approval_then_external_call_pattern':   approval_then_external_call_pattern,
            'approval_then_state_mutation_pattern':  approval_then_state_mutation_pattern,
            'control_flow_complexity_score':         control_flow_complexity_score,
            **slither_feats
        }
    except Exception:
        return zeros

print('Cell 4 contract extractor defined.')


In [5]:
# Cell 5 — Load raw data and resume from checkpoint
df = pd.read_csv(f'{DRIVE_BASE}/data/raw_contract_data.csv')

if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH) as f:
        completed = json.load(f)
    done_addrs = set(r['address'] for r in completed)
    print(f'Resumed: {len(completed)} done')
else:
    completed  = []
    done_addrs = set()
    print('Starting fresh')

remaining = df[~df['address'].isin(done_addrs)]
print(f'Rows remaining:             {len(remaining)}')
print(f'Verified remaining:         {remaining["is_verified"].sum()} '
      f'(Slither runs on these — ~120s max each)')


In [6]:
# Cell 6 — Run feature extraction with checkpoint every 100 rows
errors         = []
slither_ok     = 0
slither_failed = 0

for i, (_, row) in enumerate(remaining.iterrows()):
    try:
        feats            = extract_contract_features(row)
        feats['address'] = row['address']
        feats['label']   = row['label']

        if row['is_verified'] == 1:
            if feats['slither_warning_count_total'] > 0:
                slither_ok += 1
            else:
                slither_failed += 1

        completed.append(feats)
    except Exception as e:
        errors.append({'address': row['address'], 'error': str(e)})

    if (i + 1) % 100 == 0:
        with open(CHECKPOINT_PATH, 'w') as f:
            json.dump(completed, f)
        print(f'  Checkpoint: {len(completed)} done | '
              f'{len(errors)} errors | '
              f'slither ok/failed: {slither_ok}/{slither_failed}')

with open(CHECKPOINT_PATH, 'w') as f:
    json.dump(completed, f)
print(f'Complete: {len(completed)} rows | {len(errors)} errors')
print(f'Slither: {slither_ok} succeeded | {slither_failed} returned zeros')


In [7]:
# Cell 7 — Assemble, clean, validate
with open(CHECKPOINT_PATH) as f:
    completed = json.load(f)

df_features = pd.DataFrame(completed)
df_features = df_features[['address', 'label'] + FEATURE_COLS]
df_features = df_features.replace([float('inf'), float('-inf')], 0)
df_features = df_features.fillna(0)

print('=== VALIDATION ===')
TOTAL  = len(df_features)
PHISH  = (df_features['label'] == 1).sum()
BENIGN = (df_features['label'] == 0).sum()
assert df_features.shape[1] == 23,  f'Wrong column count: {df_features.shape[1]}'
assert PHISH == BENIGN,             f'Imbalanced: phishing={PHISH} benign={BENIGN}'
assert TOTAL >= 1000,               f'Too few rows: {TOTAL}'
print(f'Rows: {TOTAL} | Phishing: {PHISH} | Benign: {BENIGN}')

assert df_features.isnull().sum().sum() == 0,          'Nulls found'
assert not np.isinf(df_features[FEATURE_COLS].values).any(), 'Inf values found'
assert df_features['is_verified'].isin([0, 1]).all(),  'is_verified not binary'

unverified_nonzero = (
    df_features[df_features['is_verified'] == 0]['slither_warning_count_total'] > 0
).sum()
assert unverified_nonzero == 0, 'Unverified contracts have non-zero Slither features'

for col in [c for c in FEATURE_COLS if c.startswith('opcode_freq')]:
    assert df_features[col].between(0, 1).all(), f'{col} out of [0,1] range'

print(f'Shape: {df_features.shape}')
print('All validation checks passed.')


In [8]:
#  Cell 8 — Save contract_features.csv and contract_feature_schema.json
df_features.to_csv(OUTPUT_PATH, index=False)
with open(SCHEMA_PATH, 'w') as f:
    json.dump(FEATURE_COLS, f)
print(f'Saved: {OUTPUT_PATH}')
print(f'Saved: {SCHEMA_PATH}')

df_check = pd.read_csv(OUTPUT_PATH)
with open(SCHEMA_PATH) as f:
    schema_check = json.load(f)

assert df_check.shape[1] == 23,  f'Column count wrong: {df_check.shape[1]}'
assert df_check['label'].value_counts()[1] == df_check['label'].value_counts()[0], \
    'Imbalanced classes in saved CSV'
assert len(schema_check) == 21,  f'Schema wrong: {len(schema_check)}'

print(f'Verified CSV: {df_check.shape}')
print(f'Verified schema: {len(schema_check)} items')
print('Notebook 03 complete.')
